In [2]:
from langchain_community.document_loaders import UnstructuredImageLoader

In [3]:
#image = Image.open("image.jpg")

loader = UnstructuredImageLoader("image.jpg")

In [4]:
#text = pytesseract.image_to_string(image)

data = loader.load()

No languages specified, defaulting to English.


In [5]:
data

[Document(metadata={'source': 'image.jpg'}, page_content='\n\nWHO IS GREATER ?\n\nVishnudas and Sambhudas once took shelter from the rain in an old, deserted house by the roadside. By the vertical and horizontal marks on their foreheads, they showed themselves to be devotees of Vishnu and Shiva respectively. They would, therefore, not speak to each other, and kept the maximum possible distance between themselves while taking shelter in the house, faces turned away from each other.\n\nWith no one to speak to, and nothing to do, how long could they keep it up? Each was anxious to prove the superiority of his own religion. After some time, Vishnudas broke the silence. “You are a Shaivite. But look, my God,\n\noP')]

In [6]:
type(data)

list

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
#Split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)
docs = text_splitter.split_documents(data)

In [9]:
print("The number of documents = ", len(docs))

The number of documents =  2


In [10]:
#loading huggingface embedding class
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings()

vector = embeddings.embed_query("hello world!")

/var/folders/49/dlfryms541x6nvjcz5yz0hh80000gn/T/ipykernel_28322/714104251.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/var/folders/49/dlfryms541x6nvjcz5yz0hh80000gn/T/ipykernel_28322/714104251.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 6757.72it/s]


In [11]:
#creating a vector database to store all the values of vector embeddings
# the vector database is named as vectorstore

from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(docs, embedding=embeddings)

In [12]:
# we want to use this vector database as retriver
# the task of retriver is to use those documents that matches the user's query

retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs = {"k" : 5})
retrieved_docs = retriever.invoke("story")

In [13]:
# printing the length of the retrieved document

print("Length of the retrived document = ", len(retrieved_docs))

Length of the retrived document =  2


In [18]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

# Use Groq API key to load the Groq llm
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key = api_key, model_name = "llama-3.3-70b-versatile")

prompt_template = """
You are a factual assistant.

Answer the question ONLY using the context below.
- Do NOT use prior knowledge
- Do NOT refuse unless context is empty
- If answer exists in context, give it directly
- Give full length answer

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

# Chain
llm_chain = prompt | llm | StrOutputParser()

In [19]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | llm_chain
)

In [20]:
#Input question
question = "How the two persons were standing?"

#answer
response = rag_chain.invoke(question)
print(response)

They were standing with their faces turned away from each other, and keeping the maximum possible distance between themselves while taking shelter in the house.
